In [19]:
# import os
# os.environ["OPENAI_API_KEY"] = "sk-or-v1-"

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv --upgrade --force-reinstall

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio-client 1.14.0 requires websockets<16.0,>=13.0, but you have websockets 16.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
langgraph-sdk 0.4.2 requires websockets<16,>=14, but you have websockets 16.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.
google-adk 1.29.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 16.0 which is incompatible.


In [3]:
!pip install langchain-text-splitters

In [4]:
import youtube_transcript_api

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_4611/3829014579.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [6]:
from youtube_transcript_api import YouTubeTranscriptApi

In [7]:
!pip uninstall youtube-transcript-api -y
!pip install --no-cache-dir youtube-transcript-api

Found existing installation: youtube-transcript-api 1.2.4
Uninstalling youtube-transcript-api-1.2.4:
  Successfully uninstalled youtube-transcript-api-1.2.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 29.1 MB/s eta 0:00:00


In [12]:
from youtube_transcript_api import YouTubeTranscriptApi

# Make sure this is just the alphanumeric ID
video_id = "LPZh9BOjkQs"

try:
    # 1. Initialize the API instance
    yt_api = YouTubeTranscriptApi()

    # 2. Fetch the transcript list
    transcript_list = yt_api.fetch(video_id, languages=["en"])

    # 3. FIX: Access chunk.text instead of chunk["text"]
    full_text = " ".join([chunk.text for chunk in transcript_list])

    print("Success! Transcript loaded:")
    #print(full_text[:500]) # Prints the first 500 charactersa$0

except Exception as e:
    print(f"Encountered an issue: {e}")

Success! Transcript loaded:


In [13]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Submit subtitle corrections at criblate.com]\nImagine you happen across a short movie script that', start=1.14, duration=2.836), FetchedTranscriptSnippet(text='describes a scene between a person and their AI assistant.', start=3.976, duration=3.164), FetchedTranscriptSnippet(text="The script has what the person asks the AI, but the AI's response has been torn off.", start=7.48, duration=5.58), FetchedTranscriptSnippet(text='Suppose you also have this powerful magical machine that can take', start=13.06, duration=3.92), FetchedTranscriptSnippet(text='any text and provide a sensible prediction of what word comes next.', start=16.98, duration=3.98), FetchedTranscriptSnippet(text='You could then finish the script by feeding in what you have to the machine,', start=21.5, duration=4.006), FetchedTranscriptSnippet(text="seeing what it would predict to start the AI's answer,", start=25.506, duration=2.862), FetchedTranscriptSnippet(te

## Step 1b - Indexing (Text Splitting)

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.create_documents([full_text])

In [15]:
len(chunks)

11

In [16]:
chunks[2]

Document(metadata={}, page_content="does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user and a hypothetical AI assistant, you add on whatever the user types in as the first part of the interaction, and then have the model repeatedly predict the next word that such a hypothetical AI assistant would say in response, and that's what's presented to the user. In doing this, the output tends to look a lot more natural if you allow it to select less likely words along the way at random. So what this means is even though the model itself is deterministic, a given prompt typically gives a different answer each time it's run. Models learn how to make these predictions by processing an enormous amount of text, typically pulled from the internet. For a standard human to read the amount of text that was used to train GPT-3, for example, if they read non-stop 24-7, it would take over 2600 years. Lar

## Step 1b - Indexing (Text Splitting)

In [22]:
from langchain_core.embeddings import Embeddings
Embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

In [25]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embeddings)

/tmp/ipykernel_4611/3283581948.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
vector_store.index_to_docstore_id

{0: 'c0e0ded6-f3c0-49c7-bbb6-a53b5d84643f',
 1: 'cb755621-70d9-4ed2-b2b0-fb1a26aa858d',
 2: '9e112776-81dd-44fb-bfc3-9b24d929387c',
 3: 'ef3d61fe-6f8a-46cc-9412-1d7827bd10a3',
 4: '25bb81a8-a0e4-4d81-8bc6-401718b02819',
 5: 'aa8894b3-4c0b-4b71-9471-38c6c25a836d',
 6: 'a8f1eb50-6431-4e8e-a325-a5e7e8f886a3',
 7: '911b2d0c-577d-45b5-a66e-1a96621c098c',
 8: 'b14d5a32-7221-448b-b43d-6d28cdce36e6',
 9: '41efa794-6b26-4bb9-8630-3449b952bffd',
 10: 'eb681a87-fe2f-49e9-b901-99a6b311bc2f'}

In [27]:
vector_store.get_by_ids(['41efa794-6b26-4bb9-8630-3449b952bffd'])

[Document(id='41efa794-6b26-4bb9-8630-3449b952bffd', metadata={}, page_content="probability for every possible next word. Although researchers design the framework for how each of these steps work, it's important to understand that the specific behavior is an emergent phenomenon based on how those hundreds of billions of parameters are tuned during training. This makes it incredibly challenging to determine why the model makes the exact predictions that it does. What you can see is that when you use large language model predictions to autocomplete a prompt, the words that it generates are uncannily fluent, fascinating, and even useful. If you're a new viewer and you're curious about more details on how transformers and attention work, boy do I have some material for you. One option is to jump into a series I made about deep learning, where we visualize and motivate the details of attention and all the other steps in a transformer. Also, on my second channel I just posted a talk I gave 

## Step 2 - Retrieval

In [28]:
from re import search
retrieval = vector_store.as_retriever(search_type="similarity", search_kwargs ={'k':4})

In [29]:
retrieval

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7943a8a1d5e0>, search_kwargs={'k': 4})

In [31]:
retrieval.invoke("what is LLM")

[Document(id='b14d5a32-7221-448b-b43d-6d28cdce36e6', metadata={}, page_content="typically also include a second type of operation known as a feed-forward neural network, and this gives the model extra capacity to store more patterns about language learned during training. All of this data repeatedly flows through many different iterations of these two fundamental operations, and as it does so, the hope is that each list of numbers is enriched to encode whatever information might be needed to make an accurate prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this sequence, which now has had a chance to be influenced by all the other context from the input text, as well as everything the model learned during training, to produce a prediction of the next word. Again, the model's prediction looks like a probability for every possible next word. Although researchers design the framework for how each of these steps work, it's im

## Step 3 - Augmentation

In [43]:
llm = ChatOpenAI(base_url="https://openrouter.ai/api/v1",
    model='openai/gpt-4o-mini', temperature=0.2)

In [36]:
prompt = PromptTemplate(
    template = """
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [53]:
question = "What operation makes transformers unique?"
retrieved_docs = retrieval.invoke(question)

In [54]:
retrieved_docs

[Document(id='911b2d0c-577d-45b5-a66e-1a96621c098c', metadata={}, page_content='step inside a transformer, and most other language models for that matter, is to associate each word with a long list of numbers. The reason for this is that the training process only works with continuous values, so you have to somehow encode language using numbers, and each of these lists of numbers may somehow encode the meaning of the corresponding word. What makes transformers unique is their reliance on a special operation known as attention. This operation gives all of these lists of numbers a chance to talk to one another and refine the meanings they encode based on the context around, all done in parallel. For example, the numbers encoding the word bank might be changed based on the context surrounding it to somehow encode the more specific notion of a riverbank. Transformers typically also include a second type of operation known as a feed-forward neural network, and this gives the model extra cap

In [56]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"step inside a transformer, and most other language models for that matter, is to associate each word with a long list of numbers. The reason for this is that the training process only works with continuous values, so you have to somehow encode language using numbers, and each of these lists of numbers may somehow encode the meaning of the corresponding word. What makes transformers unique is their reliance on a special operation known as attention. This operation gives all of these lists of numbers a chance to talk to one another and refine the meanings they encode based on the context around, all done in parallel. For example, the numbers encoding the word bank might be changed based on the context surrounding it to somehow encode the more specific notion of a riverbank. Transformers typically also include a second type of operation known as a feed-forward neural network, and this gives the model extra capacity to store more patterns about language learned during training. All of\n\n

In [57]:
final_prompt = prompt.invoke({'context': context_text, 'question': question})

In [58]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      step inside a transformer, and most other language models for that matter, is to associate each word with a long list of numbers. The reason for this is that the training process only works with continuous values, so you have to somehow encode language using numbers, and each of these lists of numbers may somehow encode the meaning of the corresponding word. What makes transformers unique is their reliance on a special operation known as attention. This operation gives all of these lists of numbers a chance to talk to one another and refine the meanings they encode based on the context around, all done in parallel. For example, the numbers encoding the word bank might be changed based on the context surrounding it to somehow encode the more specific notion of a riverbank. Transformers typically als

## Step 4 - Generation

In [59]:
answer = llm.invoke(final_prompt)
print(answer.content)

The operation that makes transformers unique is known as attention.


In [51]:
answer

AIMessage(content="I don't know.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 808, 'total_tokens': 812, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.0001236, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0001236, 'upstream_inference_prompt_cost': 0.0001212, 'upstream_inference_completions_cost': 2.4e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_42e6cfce1b', 'id': 'gen-1782455900-ebAq263o0hC7uNoy7ZLH', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f02a6-99ed-7740-9a3e-58f319f485f1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 808, 'output_tokens': 4, 'total_tokens': 812, 'inp

## Building a Chain

In [60]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [61]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [64]:
parallel_chain = RunnableParallel({
    'context': retrieval | RunnableLambda(format_docs),
    'question':RunnablePassthrough()
})

In [65]:
parallel_chain.invoke('explain me about Transformer')

{'context': "details of attention and all the other steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago about this topic for the company TNG in Munich. Sometimes I actually prefer the content I make as a casual talk rather than a produced video, but I leave it up to you which one of these feels like the better follow-on.\n\nstep inside a transformer, and most other language models for that matter, is to associate each word with a long list of numbers. The reason for this is that the training process only works with continuous values, so you have to somehow encode language using numbers, and each of these lists of numbers may somehow encode the meaning of the corresponding word. What makes transformers unique is their reliance on a special operation known as attention. This operation gives all of these lists of numbers a chance to talk to one another and refine the meanings they encode based on the context around, all done in parallel. For 

In [66]:
parser = StrOutputParser()

In [67]:
main_chain = parallel_chain | prompt | llm | parser

In [68]:
main_chain.invoke('can you summrize the video')

"The video discusses the workings of transformers, focusing on the attention mechanism and the feed-forward neural network, which enhance the model's ability to learn language patterns. It explains how data flows through these operations to predict the next word in a sequence, resulting in fluent and useful outputs. The speaker mentions the complexity of understanding the model's predictions due to the tuning of numerous parameters during training. Additionally, the speaker offers resources for viewers interested in learning more about transformers and attention, including a series on deep learning and a recent talk given in Munich."

In [70]:
!pip install grandalf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.3 MB/s eta 0:00:00


In [73]:
import grandalf
print("Grandalf installed successfully!")

Grandalf installed successfully!


In [74]:
graph = main_chain.get_graph()
graph.print_ascii()

ImportError: Install grandalf to draw graphs: `pip install grandalf`.